# Polyester T-Shirt LCA

Cradle-to-grave carbon footprint of one polyester t-shirt, same approach as the PET bottle: Brightway for the calculation, BAFU for the background.

This one ends up as a hybrid model rather than a fully linked one. BAFU is a Swiss inventory, so it plausibly has basic materials, energy, transport, and waste treatment, but it's unlikely to have detailed textile processes like weaving, knitting, dyeing, or garment sewing as their own activities, those are specialised textile-sector datasets a general national database usually doesn't carry. Rather than guess that a matching BAFU activity exists and risk the model silently using the wrong one, I split this into two parts: search BAFU for the stages that plausibly do exist there (polyester or PET fibre material, transport, end-of-life), and link to those with real technosphere exchanges the same way the PET notebook does. Then, for the stages BAFU almost certainly doesn't cover (fibre-to-fabric processing, dyeing and finishing, garment assembly, the use phase), enter literature-based emission factors directly as `biosphere3` exchanges, the same trick used in the glass bottle notebook.

Mixing technosphere links where the background data exists with literature factors where it doesn't is a pretty normal move in a screening LCA, just worth saying explicitly in any writeup.

Functional unit: one polyester t-shirt, about 180 g, cradle-to-grave, including an assumed number of washes over its life.

What's in here:
1. Project setup
2. Import BAFU (ecoSpold1), same debugged procedure as the PET bottle notebook
3. Search BAFU for the stages it plausibly covers, and record the picks
4. Literature emission factors for the stages BAFU doesn't cover
5. Build the foreground activity, combining technosphere links and direct biosphere exchanges
6. Run the LCA
7. Contribution breakdown by stage
8. Monte Carlo

## 1. Project setup

This lives in its own project (`Polyester T-Shirt LCA`) so it doesn't interfere with the PET bottle or glass bottle ones. Every Brightway project keeps its own copy of every database, so BAFU has to be imported again here even though it's already sitting in the PET bottle project.

In [1]:
import brightway2 as bw

bw.projects.set_current("Polyester T-Shirt LCA")
bw.bw2setup()  # installs biosphere3 and LCIA methods, skip if already run for this project

print(list(bw.databases))

Biosphere database already present!!! No setup is needed
['biosphere3', 'BAFU', 'Polyester T-Shirt']


## 2. Import BAFU (ecoSpold1)

Same import procedure worked out in the PET bottle notebook: clean the `None` fields before hashing, run bw2io's standard strategies, fix BAFU's older category naming so it lines up with `biosphere3`, then drop whatever's still unlinked (mostly BAFU processes pointing at ecoinvent activities that aren't in this project) and write the database.

Update the path below to the extracted ecoSpold1 folder.

In [2]:
import bw2io as bi

ECOSPOLD_PATH = "/Users/abhi/ecoSpold files"

importer = bi.SingleOutputEcospold1Importer(ECOSPOLD_PATH, "BAFU")

Extracting XML data from 11947 datasets


/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(

Extracted 11947 datasets in 26.20 seconds


In [3]:
# Fix None fields before hashing, same issue found when importing for the PET bottle model
fields_to_clean = ["name", "categories", "unit", "reference product", "location"]

for ds in importer.data:
    for field in fields_to_clean:
        if ds.get(field) is None:
            ds[field] = ""
    for exc in ds.get("exchanges", []):
        for field in fields_to_clean:
            if exc.get(field) is None:
                exc[field] = ""

In [4]:
importer.apply_strategies()
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: normalize_units
Applying strategy: assign_only_product_as_production
Applying strategy: clean_integer_codes
Applying strategy: drop_unspecified_subcategories
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: update_ecoinvent_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: link_technosphere_by_activity_hash
Applied 11 strategies in 19.21 seconds
Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
295213 unlinked exchanges
  Type biosphere: 2679 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 295213)

In [5]:
# BAFU's older category naming, mapped directly to biosphere3's category names
CATEGORY_FIX = {
    ("emissions to air",): ("air",),
    ("emissions to air", "high. pop."): ("air", "urban air close to ground"),
    ("emissions to air", "low. pop."): ("air", "non-urban air or from high stacks"),
    ("emissions to air", "stratosphere + troposphere"): ("air", "lower stratosphere + upper troposphere"),
    ("emissions to soil",): ("soil",),
    ("emissions to soil", "agricultural"): ("soil", "agricultural"),
    ("emissions to soil", "industrial"): ("soil", "industrial"),
    ("emissions to water",): ("water",),
    ("emissions to water", "ground-"): ("water", "ground-"),
    ("emissions to water", "ocean"): ("water", "ocean"),
    ("emissions to water", "river"): ("water", "surface water"),
    ("resources",): ("natural resource",),
    ("resources", "land"): ("natural resource", "land"),
    ("resources", "in ground"): ("natural resource", "in ground"),
    ("resources", "in air"): ("natural resource", "in air"),
    ("resources", "in water"): ("natural resource", "in water"),
    ("resources", "biotic"): ("natural resource", "biotic"),
}

fixed_count = 0
for ds in importer.data:
    for exc in ds.get("exchanges", []):
        if exc.get("type") == "biosphere":
            cats = tuple(exc.get("categories") or ())
            if cats in CATEGORY_FIX:
                exc["categories"] = CATEGORY_FIX[cats]
                fixed_count += 1

print(f"Rewrote categories on {fixed_count} exchanges")

Rewrote categories on 268970 exchanges


In [6]:
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
114060 unlinked exchanges
  Type biosphere: 1155 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 114060)

Check what's still unlinked before dropping it. If the biosphere count is small, the fix above worked, and the technosphere count will probably show the same handful of ecoinvent-only references seen in the PET bottle import (electricity mixes, natural gas production, that sort of thing).

In [7]:
unlinked = list(importer.unlinked)
bio_unlinked = [u for u in unlinked if u.get("type") == "biosphere"]
tech_unlinked = [u for u in unlinked if u.get("type") == "technosphere"]

print(f"{len(bio_unlinked)} unique unlinked biosphere flows")
for u in bio_unlinked[:20]:
    print(u.get("name"), "|", u.get("categories"), "|", u.get("unit"))

print(f"\n{len(tech_unlinked)} unique unlinked technosphere")
for u in tech_unlinked[:24]:
    print(u.get("name"), "|", u.get("location"), "|", u.get("unit"))

1155 unique unlinked biosphere flows
NMVOC, non-methane volatile organic compounds, unspecified origin | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'non-urban air or from high stacks') | kilogram
Occupation, industrial area, built up | ('natural resource', 'land') | square meter-year
Transformation, to industrial area, built up | ('natural resource', 'land') | square meter
Particulates, < 2.5 um | ('air', 'urban air close to ground') | kilogram
Copper | ('air', 'urban air close to ground') | kilogram
Particulates, < 2.5 um | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'urban air close to ground') | kilogram
Selenium | ('air', 'non-urban air or from high stacks') | kilogram
Selenium | ('air', 'urban air close to ground') | kilogram
Nickel | ('air', 'non-urban air or from high stacks') | kilogram
Zinc | ('air', 'non-urban air or from high stacks') | kilogram
Nickel | ('air', 'urban air close to ground') | kilogram
Zinc | ('air'

In [8]:
importer.drop_unlinked(i_am_reckless=True)
importer.statistics()

Applying strategy: drop_unlinked
Applied 1 strategies in 0.77 seconds
11947 datasets
306003 exchanges
0 unlinked exchanges
  


(11947, 306003, 0)

In [10]:
importer.write_database()
print(list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:55:32
  Finished: 08/06/2026 11:55:55
  Total time elapsed: 00:00:22
  CPU %: 80.50
  Memory %: 1.45
Created database: BAFU
['biosphere3', 'BAFU', 'Polyester T-Shirt']


## 3. Search BAFU for the stages it plausibly covers

Run each search and actually look at what comes back before picking anything. BAFU's naming won't always match these search terms, worth trying a few variants if the first one comes up empty. If a pick isn't obvious from the results, that's a good moment to check with someone rather than guess.

In [11]:
ei = bw.Database("BAFU")

# Polyester fibre material. Polyester is chemically PET, so BAFU's PET-related
# activities (the same ones used in the PET bottle model) are the most likely match.
for r in ei.search("polyethylene terephthalate"):
    print(r, "|", r["location"])

'Fleece production, polyethylene terephthalate' (kilogram, RER, ['plastics']) | RER
'Polyethylene terephthalate, granulate, amorphous, at plant' (kilogram, RER, ['plastics', 'thermoplasts']) | RER
'Polyethylene terephthalate, granulate, bottle grade, at plant' (kilogram, RER, ['plastics', 'thermoplasts']) | RER
'xx Recycling PET' (kilogram, RER, ['waste treatment, obsolete', 'recycling, obsolete']) | RER
'Ethylene glycol, at plant' (kilogram, RER, ['chemicals', 'organic']) | RER


In [12]:
# Fibre or filament specific search, in case BAFU has a textile fibre process
# distinct from the bottle-grade granulate used in the PET bottle model
for r in ei.search("fibre"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("filament"):
    print(r, "|", r["location"])

'Glass fibre, at plant' (kilogram, RER, ['glass', 'construction']) | RER
'Packing, fibre cement products' (kilogram, CH, ['Others']) | CH
'Fibre cement corrugated slab, at plant' (kilogram, CH, ['construction', 'coverings']) | CH
'Fibre cement roof slate, at plant' (kilogram, CH, ['construction', 'coverings']) | CH
'Fibre cement facing tile, at plant' (kilogram, CH, ['construction', 'coverings']) | CH
'Gypsum fibre board, at plant' (kilogram, CH, ['construction', 'coverings']) | CH
'Gypsum fibre board, at plant' (square meter, DE, ['construction materials', 'gypsum materials']) | DE
'xx Cellulose fibre, inclusive blowing in, at plant' (kilogram, CH, ['material, obsolete', 'construction, obsolete\\insulation, obsolete']) | CH
'xx Kenaf stalks, from fibre production, at farm' (kilogram, IN, ['material, obsolete', 'agricultural, obsolete\\plant production, obsolete']) | IN
'Glass fibre reinforced plastic, polyamide, injection moulding, at plant' (kilogram, RER, ['plastics', 'thermoplasts'

In [13]:
# Transport, reuse the same fleet-average search from the PET bottle model
for r in ei.search("transport lorry"):
    print(r, "|", r["location"])

'Transport, municipal waste collection, lorry 21t' (ton kilometer, CH, ['transport systems', 'road']) | CH
'Transport, freight, lorry, fleet average' (ton kilometer, RER, ['transport systems', 'road\\Transformation']) | RER
'Transport, freight, lorry, fleet average' (ton kilometer, CH, ['transport systems', 'road\\Transformation']) | CH
'Lorry 16t' (unit, RER, ['transport systems', 'road\\operations\\infrastructure']) | RER
'Lorry 28t' (unit, RER, ['transport systems', 'road\\operations\\infrastructure']) | RER
'Lorry 40t' (unit, RER, ['transport systems', 'road\\operations\\infrastructure']) | RER
'Transport, freight, lorry, diesel, fleet average, long haul' (ton kilometer, RER, ['transport systems', 'road\\Transformation']) | RER
'Transport, freight, lorry, diesel, fleet average, regional delivery' (ton kilometer, CH, ['transport systems', 'road\\Transformation']) | CH
'Transport, freight, lorry, diesel, fleet average, regional delivery' (ton kilometer, RER, ['transport systems', 'ro

In [14]:
# End of life: textile-specific disposal if it exists, otherwise a general
# municipal waste or landfill process as a proxy, the same technique used
# for the PET bottle's polyethylene-as-proxy disposal picks
for r in ei.search("textile"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("used clothing"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("municipal solid waste incineration"):
    print(r, "|", r["location"])

'Textile refinement, cotton' (kilogram, GLO, ['textiles']) | GLO
'xx Textile, kenaf, at plant' (kilogram, IN, ['material, obsolete', 'textiles, obsolete']) | IN
'xx Textile, jute, at plant' (kilogram, IN, ['material, obsolete', 'textiles, obsolete']) | IN
'Textile, woven cotton, at plant' (kilogram, GLO, ['textiles']) | GLO
'Flooring, tufted carpet with textile fabric backs, at regional storage' (square meter, CH, ['flooring', 'flooring kbob']) | CH
'Disposal, tufted carpet with textile fabric backs, as building waste' (square meter, CH, ['waste management', 'building demolition']) | CH
'Weeving, cotton' (kilogram, GLO, ['textiles']) | GLO
'Weeving, bast fibres' (kilogram, IN, ['textiles']) | IN
'Flax culture, flax seeds' (kilogram, FR, ['construction materials', 'flax insulation']) | FR
'Flax culture, flax straw' (kilogram, FR, ['construction materials', 'flax insulation']) | FR
'Jute fibers, recycled' (kilogram, FR, ['construction materials', 'flax insulation']) | FR
'Flooring, needl

## 4. Emission factors for the stages BAFU doesn't cover

Fibre-to-fabric processing, dyeing and finishing, garment assembly, and washing are specialised textile-industry processes that a general national inventory like BAFU almost certainly doesn't carry at this level of detail, so these use literature factors instead of a technosphere link. The values below are illustrative, drawn from typical textile LCA literature (Quantis and Higg Index style studies, roughly), and should be swapped for numbers from a specific source before this feeds into anything real.

Same as the glass bottle notebook, each factor stays as a named, commented variable, so the assumption behind every number is visible and changeable on its own.

In [15]:
# ---- Physical parameters ----
TSHIRT_MASS_KG = 0.180   # a typical polyester t-shirt weighs about 180 g
DIST_KM = 100             # factory to retailer distance, same assumption as the other notebooks
N_WASHES = 20             # assumed number of machine washes over the garment's useful life

# ---- End-of-life split ----
EoL_INCINERATION = 0.50
EoL_LANDFILL = 0.50
# (textile recycling rates for polyester garments remain low in most regions,
#  so it is not included as a separate pathway in this simplified model)

# ---- Emission factors, in kg CO2-eq per kg of fabric unless noted ----

# Fibre-to-fabric: spinning polyester filament into yarn, then knitting or weaving into fabric
EF_SPIN_KNIT = 3.5

# Dyeing and finishing: hot water dye baths, chemical processing, drying
EF_DYE_FINISH = 5.5

# Garment assembly: cutting and sewing, mostly electricity for machines
EF_ASSEMBLY = 0.3

# Use phase: kg CO2-eq per wash cycle (washing machine electricity and water heating,
# apportioned to this one garment within a full laundry load)
EF_PER_WASH = 0.05

print("Emission factors set.")

Emission factors set.


## 5. Build the foreground activity

This activity combines two kinds of exchanges. Technosphere exchanges (once the picks from section 3 are filled in) link to real BAFU activities for fibre material, transport, and end-of-life, same technique as the PET bottle model. Biosphere exchanges enter the literature-based stages directly as `'Carbon dioxide, fossil'`, same technique as the glass bottle model, since the IPCC GWP100 characterisation factor for this flow is exactly 1.

In [16]:
co2_candidates = [f for f in bw.Database("biosphere3")
                  if f["name"] == "Carbon dioxide, fossil" and f["categories"] == ("air",)]
co2_flow = co2_candidates[0]
print(co2_flow)

'Carbon dioxide, fossil' (kilogram, None, ('air',))


In [17]:
spin_knit_kgco2 = TSHIRT_MASS_KG * EF_SPIN_KNIT
dye_finish_kgco2 = TSHIRT_MASS_KG * EF_DYE_FINISH
assembly_kgco2 = TSHIRT_MASS_KG * EF_ASSEMBLY
use_phase_kgco2 = N_WASHES * EF_PER_WASH

print(f"Spinning/knitting:  {spin_knit_kgco2:.4f} kg CO2-eq")
print(f"Dyeing/finishing:   {dye_finish_kgco2:.4f} kg CO2-eq")
print(f"Garment assembly:   {assembly_kgco2:.4f} kg CO2-eq")
print(f"Use phase ({N_WASHES} washes): {use_phase_kgco2:.4f} kg CO2-eq")

Spinning/knitting:  0.6300 kg CO2-eq
Dyeing/finishing:   0.9900 kg CO2-eq
Garment assembly:   0.0540 kg CO2-eq
Use phase (20 washes): 1.0000 kg CO2-eq


In [18]:
DB_NAME = "Polyester T-Shirt"

if DB_NAME in bw.databases:
    del bw.databases[DB_NAME]

exchanges = [
    {"input": (DB_NAME, "polyester_tshirt"), "amount": 1, "unit": "piece", "type": "production"},

    # ---- Literature-factor stages, direct biosphere exchanges ----
    {"input": co2_flow.key, "amount": spin_knit_kgco2, "unit": "kilogram", "type": "biosphere"},
    {"input": co2_flow.key, "amount": dye_finish_kgco2, "unit": "kilogram", "type": "biosphere"},
    {"input": co2_flow.key, "amount": assembly_kgco2, "unit": "kilogram", "type": "biosphere"},
    {"input": co2_flow.key, "amount": use_phase_kgco2, "unit": "kilogram", "type": "biosphere"},

    # ---- BAFU-linked stages, technosphere exchanges ----
    # Uncomment each line once the matching pick from section 3 has been confirmed
    # {"input": polyester_fibre.key, "amount": TSHIRT_MASS_KG, "unit": "kilogram", "type": "technosphere"},
    # {"input": truck.key, "amount": TSHIRT_MASS_KG * DIST_KM / 1000, "unit": "ton kilometer", "type": "technosphere"},
    # {"input": incineration.key, "amount": TSHIRT_MASS_KG * EoL_INCINERATION, "unit": "kilogram", "type": "technosphere"},
    # {"input": landfill.key, "amount": TSHIRT_MASS_KG * EoL_LANDFILL, "unit": "kilogram", "type": "technosphere"},
]

foreground = bw.Database(DB_NAME)
foreground.write({
    (DB_NAME, "polyester_tshirt"): {
        "name": "Polyester t-shirt, cradle-to-grave",
        "reference product": "Polyester t-shirt, cradle-to-grave",
        "unit": "piece",
        "location": "CH",
        "type": "process",
        "exchanges": exchanges,
    }
})

print("Database written:", list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:58:15
  Finished: 08/06/2026 11:58:15
  Total time elapsed: 00:00:00
  CPU %: 41.50
  Memory %: 0.72
Database written: ['biosphere3', 'BAFU', 'Polyester T-Shirt']


Note to self: until the BAFU-linked exchanges above are uncommented, this model only accounts for spinning/knitting, dyeing/finishing, assembly, and the use phase. The fibre material, transport, and end-of-life aren't in yet, so the score in section 6 will understate the real footprint until those lines are filled in and this cell is rerun.

## 6. Run the LCA

Same method as the other two notebooks, so all three products land on equal footing for comparison.

In [19]:
tshirt = bw.get_activity((DB_NAME, "polyester_tshirt"))

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

lca = bw.LCA({tshirt: 1}, method)
lca.lci()
lca.lcia()

print(f"GWP 100a: {lca.score:.4f} kg CO2-eq per t-shirt")

GWP 100a: 2.6740 kg CO2-eq per t-shirt


## 7. Contribution breakdown by stage

Textile LCAs commonly find that dyeing and finishing, and sometimes the use phase (repeated washing over the garment's life), dominate the footprint rather than the fibre material itself. This checks whether that pattern holds here, it'll update automatically once the BAFU-linked stages are added.

In [20]:
stages = {
    "Spinning/knitting": spin_knit_kgco2,
    "Dyeing/finishing": dye_finish_kgco2,
    "Garment assembly": assembly_kgco2,
    "Use phase (washing)": use_phase_kgco2,
    # Add the BAFU-linked stages here once available, for example:
    # "Fibre material": TSHIRT_MASS_KG,  # replace with the actual BAFU exchange's contribution
}

total = sum(stages.values())
print(f"Total (literature-factor stages only): {total:.4f} kg CO2-eq per t-shirt\n")
for name, value in stages.items():
    share = 100 * value / total
    print(f"{name:<22} {value:.4f} kg CO2-eq  ({share:5.1f}%)")

Total (literature-factor stages only): 2.6740 kg CO2-eq per t-shirt

Spinning/knitting      0.6300 kg CO2-eq  ( 23.6%)
Dyeing/finishing       0.9900 kg CO2-eq  ( 37.0%)
Garment assembly       0.0540 kg CO2-eq  (  2.0%)
Use phase (washing)    1.0000 kg CO2-eq  ( 37.4%)


## 8. Monte Carlo uncertainty analysis

The single `GWP 100a` score above is one point estimate, it hides how much the result could vary given real uncertainty in every exchange behind it. Monte Carlo reruns the calculation many times, resampling every exchange from its uncertainty distribution each time, and returns a spread instead of one number. A mean plus a confidence interval is more honest to report than a single figure, especially in a comparison between products.

Two things to know before running this: `bw2calc`'s Monte Carlo solver can raise `ValueError: 'scipy.sparse.linalg.cgs' called with invalid atol='legacy'` on newer `scipy` versions, if that happens, run `conda install "scipy<1.12"` in the `ab` environment, restart the kernel, and re-run this cell. And Monte Carlo only produces a spread if the exchanges being sampled actually carry an uncertainty distribution, background BAFU exchanges may carry some from the original ecoSpold1 data, but the foreground exchanges written here (the t-shirt's own amounts) don't have uncertainty set by default, so they stay fixed across iterations unless it's added explicitly, for example through Activity Browser's Uncertainty tab.

In [21]:
method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

mc = bw.MonteCarloLCA({tshirt: 1}, method)
n_iterations = 1000
scores = [next(mc) for _ in range(n_iterations)]

import numpy as np

print(f"Iterations: {n_iterations}")
print(f"Mean:   {np.mean(scores):.4f} kg CO2-eq per t-shirt")
print(f"Median: {np.median(scores):.4f} kg CO2-eq per t-shirt")
print(f"Std dev: {np.std(scores):.4f}")
print(f"5th to 95th percentile: {np.percentile(scores, 5):.4f} to {np.percentile(scores, 95):.4f} kg CO2-eq")

Iterations: 1000
Mean:   2.6740 kg CO2-eq per t-shirt
Median: 2.6740 kg CO2-eq per t-shirt
Std dev: 0.0000
5th to 95th percentile: 2.6740 to 2.6740 kg CO2-eq


In [ ]:
print(list(bw.databases))
print(len(bw.Database("BAFU")))

['biosphere3', 'BAFU', 'Polyester T-Shirt']
11947
